# RAG Sprint 1 – Semantic Search Engine for "Homebuyer's Guide"

This notebook builds the **Retrieval** layer of a RAG system, step-by-step:

1. **Document Loading** – Reading the PDF and converting it into accessible text.
2. **Text Chunking** – Splitting the full text into smaller chunks with an overlap to preserve context.
3. **Embeddings** – Converting each text chunk into a vector embedding using the `gemini-embedding-001` model.
4. **Pinecone Indexing** – Storing and organizing the vectors inside a dedicated vector database.
5. **Semantic Search** – Running queries and retrieving the most relevant text segments based on semantic meaning rather than just keyword matching.

> Each step is fully documented within a Markdown cell explaining what happens and why, ensuring a deep understanding of the system's underlying logic.

## Step 2 – Document Loading and Text Chunking

### Why do we split text into chunks?
Embedding and Retrieval models perform significantly better on short, focused segments rather than an entire document. Additionally, models have input length limitations. Breaking down the text into smaller portions ("chunks") ensures higher accuracy and efficiency.

### What is `chunk_overlap`?
To prevent a sentence or an idea from being cut off in the middle between two consecutive chunks, we keep a small overlap (a set number of characters or tokens) between them. This guarantees that context or information lying on the boundary of a chunk is preserved entirely in at least one of the segments.

In [ ]:
# === Setup cell – run this first ===
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VENV_PYTHON = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"

if not str(sys.executable).endswith(r".venv\Scripts\python.exe"):
    print("⚠️  Wrong kernel! Select: Python (RAG-system)")
    print("   Current Python:", sys.executable)
    print("   Required Python: ", VENV_PYTHON)
    raise SystemExit("Switch the kernel and then Restart Kernel → Run All")

def _pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "-q"])

try:
    import pip_system_certs.bootstrap
except ImportError:
    print("Installing pip-system-certs...")
    _pip_install("pip-system-certs")
    import pip_system_certs.bootstrap

import os

try:
    from dotenv import load_dotenv
    from pypdf import PdfReader
except ImportError:
    print("Installing project packages...")
    _pip_install("-r", str(PROJECT_ROOT / "requirements.txt"))
    from dotenv import load_dotenv
    from pypdf import PdfReader

load_dotenv(PROJECT_ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("GEMINI_API_KEY:", "Yes ✓" if GEMINI_API_KEY else "No ✗ – create a .env file")
print("PINECONE_API_KEY:", "Yes ✓" if PINECONE_API_KEY else "No ✗ – create a .env file")

PDF_PATH = PROJECT_ROOT / "data" / "cs229-notes2.pdf"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
INDEX_NAME = "cs229-notes2"

print("\nPDF:", PDF_PATH)
print("PDF exists:", "Yes ✓" if PDF_PATH.exists() else "No ✗")

Python: C:\Users\WIN 11\Desktop\תהילה\יד\AI\RAG-system\.venv\Scripts\python.exe
Project root: C:\Users\WIN 11\Desktop\תהילה\יד\AI\RAG-system
GEMINI_API_KEY: כן ✓
PINECONE_API_KEY: כן ✓

PDF: C:\Users\WIN 11\Desktop\תהילה\יד\AI\RAG-system\data\apartment_buyer_guide.pdf
PDF קיים: כן ✓


In [ ]:
import re


def clean_text(text):
    """Clean text extracted from PDF: compress long sequences of whitespace and blank lines
    (the PDF contains many blank lines that add noise to embeddings)."""
    text = re.sub(r"[ \t]+", " ", text)      # whitespace sequence -> single space
    text = re.sub(r"\n\s*\n+", "\n", text)   # multiple blank lines -> one line
    return text.strip()


def load_pdf(pdf_path):
    """Read a PDF and return:
    - full_text: all document text as one cleaned string
    - char_page: a list mapping each character position i to the page it came from (for metadata)
    """
    reader = PdfReader(str(pdf_path))
    full_text = ""
    char_page = []
    for page_num, page in enumerate(reader.pages, start=1):
        page_text = clean_text(page.extract_text() or "") + "\n"
        full_text += page_text
        char_page.extend([page_num] * len(page_text))
    return full_text, char_page


full_text, char_page = load_pdf(PDF_PATH)

print(f"Number of pages: {char_page[-1] if char_page else 0}")
print(f"Total characters in text: {len(full_text):,}")
print("\n--- Preview (first 300 characters) ---")
print(full_text[:300])

מספר עמודים: 18
סך תווים בטקסט: 19,998

--- תצוגה מקדימה (300 תווים ראשונים) ---
1 
מדריך לרוכש דירה
2 
תוכן העניינים 
מבוא : מה צריך לבדוק לפני שרוכשים דירה?1 
פרק1 : הדירה ואזור המגוריםבחירת4 
פרק2 : בודקיםמה לפני הרכישה6 
פרק3 : יכרון דברים וחוזה המכרז8 
פרק4 : מהלך הבנייה11 
פרק5 : הבטחת הכספים12 
פרק6 : קבלת הדירה ואחריות מוכר הדירה15 
פרק7 : הגשת תלונות למשרד הבינוי והשיכו


In [ ]:
def chunk_text(text, char_page, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Split long text into chunks of size chunk_size characters, with chunk_overlap overlap.
    Each chunk receives metadata with the page where it starts.
    Returns a list of dicts: {id, text, page}.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    step = chunk_size - chunk_overlap  # how much we advance each time
    idx = 0
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_str = text[start:end].strip()
        if chunk_str:  # skip empty chunks
            page = char_page[start] if start < len(char_page) else char_page[-1]
            chunks.append({
                "id": f"chunk-{idx}",
                "text": chunk_str,
                "page": page,
            })
            idx += 1
        start += step
    return chunks

In [ ]:
chunks = chunk_text(full_text, char_page, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Created {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print("\n--- Example: the first chunk ---")
print("id:", chunks[0]["id"], "| page:", chunks[0]["page"])
print(chunks[0]["text"][:400], "...")

print("\n--- Example: a chunk from the middle of the document ---")
mid = len(chunks) // 2
print("id:", chunks[mid]["id"], "| page:", chunks[mid]["page"])
print(chunks[mid]["text"][:400], "...")

נוצרו 24 chunks (chunk_size=1000, chunk_overlap=150)

--- דוגמה: ה-chunk הראשון ---
id: chunk-0 | עמוד: 1
1 
מדריך לרוכש דירה
2 
תוכן העניינים 
מבוא : מה צריך לבדוק לפני שרוכשים דירה?1 
פרק1 : הדירה ואזור המגוריםבחירת4 
פרק2 : בודקיםמה לפני הרכישה6 
פרק3 : יכרון דברים וחוזה המכרז8 
פרק4 : מהלך הבנייה11 
פרק5 : הבטחת הכספים12 
פרק6 : קבלת הדירה ואחריות מוכר הדירה15 
פרק7 : הגשת תלונות למשרד הבינוי והשיכון17 
פרק8 : הבהרות כלליות שימוש במדריך18
3 
צעדיו הראשונים לקראת רכישה של דירהדריך זה נכתב במיוחד לר ...

--- דוגמה: chunk מאמצע המסמך ---
id: chunk-12 | עמוד: 10
רישום של רשות מקרקעי בחוזה. כמו כן, מומלץ לבדוק אם החברה המוכרת מצויה ב
. ישראל
ליווי משפטי בעת תהליך רכישת הדירה 
רצוי לשכור את שירותיו של עורך דין מתחום הנדל"ן לצורך בדיקת כל פרטי הדירה וניהול המשא ומתן 
כמו כן, רצוי לחתום על חוזה המכר בנוכחות עורך הדין שנשכר מטעמכם. זכרו להחתים את . מול המוכר
 .המוכר גם על העתק החוזה שנשאר ברשותכם
בכל הנוגע לרישום הדירה שימו לב: המוכר שוכר עורך דין מטעמו אשר נו ...


## Step 3 – Creating Embeddings with Gemini

**What is an embedding?**
An embedding is a numerical representation (vector) of text. Text with similar meaning gets vectors that are close in space.

**Why `task_type`?**
For the `gemini-embedding-001` model, it is recommended to separate:
- `RETRIEVAL_DOCUMENT` – for text stored in the knowledge base (chunks)
- `RETRIEVAL_QUERY` – for user questions (used during search)

**If you get an SSL error (common with NetFree):**
1. Make sure a `.env` file (not `.env.example`) exists with the real keys.
2. Install the NetFree security certificate on your computer.
3. If it still fails, run in the terminal: `pip install pip-system-certs` and then **Restart Kernel** in the notebook.

In [ ]:
from google import genai
from google.genai import types

# Create a Gemini client with the key from .env
# (the key is loaded in the setup cell above)
if not GEMINI_API_KEY:
    raise ValueError(
        "Missing GEMINI_API_KEY. Create a .env file in the project root:\n"
        "Copy-Item .env.example .env\n"
        "Then add the real key to .env (not to .env.example!)"
    )

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client created successfully")

Gemini client נוצר בהצלחה


In [ ]:
import time

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """Convert a list of texts to vectors, including automatic retry on rate limit."""
    for attempt in range(3):
        try:
            result = client.models.embed_content(
                model=EMBED_MODEL,
                contents=texts,
                config=types.EmbedContentConfig(
                    task_type=task_type,
                    output_dimensionality=EMBED_DIM,
                ),
            )
            return [emb.values for emb in result.embeddings]
        except Exception as exc:
            if "429" in str(exc) and attempt < 2:
                print("Rate limit – waiting 35 seconds...")
                time.sleep(35)
                continue
            raise

sample_vector = embed_texts([chunks[0]["text"]], task_type="RETRIEVAL_DOCUMENT")[0]
print(f"Vector dimension: {len(sample_vector)}")
print(f"First 3 values: {sample_vector[:3]}")

NameError: name 'chunks' is not defined

In [ ]:
# Create embeddings for all chunks (this may take a few seconds)
all_texts = [c["text"] for c in chunks]
all_vectors = embed_texts(all_texts, task_type="RETRIEVAL_DOCUMENT")

# Attach the vector to each chunk
for chunk, vector in zip(chunks, all_vectors):
    chunk["vector"] = vector

print(f"Created embeddings for {len(all_vectors)} chunks")
print(f"Example: {chunks[0]['id']} -> vector of length {len(chunks[0]['vector'])}")

נוצרו embeddings ל-24 chunks
דוגמה: chunk-0 -> וקטור באורך 768


## Step 4 – Saving to Pinecone (Vector Database)

**What is Pinecone?**
A vector database that can quickly search for the vectors most similar to a query.
In RAG, after generating embeddings for each chunk, we store them in Pinecone with metadata (text, page).

**What happens here?**
1. Connect to Pinecone with the key from `.env`
2. Create an index (if it does not already exist) with dimension 768 and metric `cosine`
3. Upsert all chunks with their vectors

In [ ]:
from pinecone import Pinecone, ServerlessSpec

if not PINECONE_API_KEY:
    raise ValueError(
        "Missing PINECONE_API_KEY. Add it to the .env file (not .env.example)."
    )

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create the index only if it does not already exist
existing_indexes = {idx.name for idx in pc.list_indexes()}
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created a new index: {INDEX_NAME}")
else:
    print(f"Index '{INDEX_NAME}' already exists – using it")

index = pc.Index(INDEX_NAME)
print("Connected to Pinecone index:", INDEX_NAME)

Index 'rag-apartment-guide' כבר קיים – משתמשים בו
מחוברים ל-Pinecone index: rag-apartment-guide


In [ ]:
# Prepare records for upsert: (id, vector, metadata)
records = [
    (
        chunk["id"],
        chunk["vector"],
        {
            "text": chunk["text"][:1000],  # Pinecone limits metadata – we store the beginning of the text
            "page": chunk["page"],
        },
    )
    for chunk in chunks
]

# upsert = insert/update vectors in the index
index.upsert(vectors=records)

stats = index.describe_index_stats()
print(f"Uploaded {len(records)} vectors")
print("Index statistics:", stats)

הועלו 24 vectors
סטטיסטיקות index: DescribeIndexStatsResponse(dimension=768, total_vector_count=24, metric='cosine', namespaces=1)


## Step 5 – Semantic Search

**How it works**
1. Take a user question
2. Create an embedding for the question with `RETRIEVAL_QUERY`
3. Send the vector to Pinecone and request the 3 closest results (`top_k=3`)
4. Check whether the results are actually relevant to the question

In [ ]:
def semantic_search(question, top_k=3):
    """Search for the top_k most relevant chunks for a question."""
    query_vector = embed_texts([question], task_type="RETRIEVAL_QUERY")[0]
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    return results.matches


def print_results(question, matches):
    print("=" * 70)
    print("Question:", question)
    print("=" * 70)
    for i, match in enumerate(matches, start=1):
        page = match.metadata.get("page", "?")
        text = match.metadata.get("text", "")
        print(f"\n#{i} | score: {match.score:.4f} | page: {page} | id: {match.id}")
        print(text[:350], "...")

In [ ]:
# 5 example questions about the guide
questions = [
    "What should you check about a building permit before buying an apartment?",
    "How do you protect the buyer's funds under the Sale Law?",
    "What happens if the seller is late in handing over the apartment?",
    "What should you check in the sales contract before signing?",
    "What is the inspection period and the seller's warranty?",
]

for q in questions:
    matches = semantic_search(q, top_k=3)
    print_results(q, matches)
    print()

שאלה: מה צריך לבדוק לגבי היתר בנייה לפני רכישת דירה?

#1 | score: 0.7832 | page: 7 | id: chunk-6
הדירה את הקרקע 
המוכרובתמורה מקבל חלק מהדירות בבניין שיבנה, עליכם לבדוק: האם רשומה הערת אזהרה על שם 
בקשר המוכרקע, ומהם תנאי הﬠִסקה בין קרקע, האם יוכל הקונה לרשום הערת אזהרה על הקרל לבין 
קרקע המקוריים.בעל הקרקע. שימו לב שהדירה העומדת לרכישה לא יוחדה למי מבעלי ה 
היתר בנייה 
שימו לב שהדירה המועמדת לרכישה תיבנה כדין. ודאו כי בידי המוכר היתר בנייה שה ...

#2 | score: 0.7371 | page: 5 | id: chunk-4
יצע דירות קיימות בבניין
נוחות המגורים שלכם. לדוגמה: דירה בקומה ראשונה, קלה לגישה אך היא קרובה יותר לקרקע וחשופה 
בדרך כלל למעבר של דיירים רבים. יש לבדוק כמה קומות יש בבניין ומהי מידת הפרטיות שמספקת כל 
דירה.
6 
 :2פרק
 בודקיםמה לפני הרכישה
מתבצעת על ידי קבלן רשום כחוקהאם הבנייה ?
של המוכר להציג בפניכם רישיון מרשום בפנקס הקבלנים. בקשו הודאו שהדירה נ ...

#3 | score: 0.7350 | page: 3 | id: chunk-1
עש באזור 
o אווירהכיווני בדירה מיקום זריחת השמש ושקיעתהו 
o ביבההס נשקפתה הדירהתוך מ 
• הפרויקט 
o קבלן ה

## Bonus – Comparing `chunk_size` and `chunk_overlap`

**Why is this important?**
Chunk size directly affects retrieval quality:
- **Large chunks** – more context, but less focused; they may dilute relevance.
- **Small chunks** – more focused, but may miss broader context.
- **High overlap** – preserves continuity between segments, but creates redundant chunks.

**What will we compare?**
| Setting | chunk_size | chunk_overlap |
|--------|-----------|---------------|
| A (default) | 1000 | 150 |
| B (small chunks) | 500 | 80 |

We will run the same test question on both settings and compare the top 3 results.
(This comparison is local with cosine similarity – without Pinecone, so no second index is created.)

In [ ]:
import numpy as np

CONFIGS = {
    "A_default": {"chunk_size": 1000, "chunk_overlap": 150},
    "B_small":   {"chunk_size": 500,  "chunk_overlap": 80},
}

TEST_QUESTION = "What happens if the seller is late in handing over the apartment?"


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def search_local(question, chunk_list, top_k=3):
    """Local search: embed the question and chunks, return top_k by cosine similarity."""
    q_vec = embed_texts([question], task_type="RETRIEVAL_QUERY")[0]
    doc_vecs = embed_texts([c["text"] for c in chunk_list], task_type="RETRIEVAL_DOCUMENT")

    scored = []
    for chunk, vec in zip(chunk_list, doc_vecs):
        scored.append((cosine_similarity(q_vec, vec), chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

In [ ]:
comparison_results = {}

for name, cfg in CONFIGS.items():
    cfg_chunks = chunk_text(full_text, char_page, **cfg)
    top = search_local(TEST_QUESTION, cfg_chunks, top_k=3)
    comparison_results[name] = top

    print("=" * 70)
    print(f"Setting {name}: chunk_size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']}")
    print(f"Number of chunks: {len(cfg_chunks)}")
    print("=" * 70)

    for i, (score, chunk) in enumerate(top, start=1):
        print(f"\n#{i} | score: {score:.4f} | page: {chunk['page']} | id: {chunk['id']}")
        print(chunk["text"][:300], "...")
    print()

Setting A_default: chunk_size=1000, overlap=150
Number of chunks: 24

#1 | score: 0.6932 | page: 10 | id: chunk-11
מה שנקבע בחוזה המכר תקף 
ה;לעניין של איחור בנסיבות שאינן בשליטת מוכר הדיר 
• , עליו לפצות את הרוכש אם המוכר איחר במסירת הדירה מעבר לחודשיים מהתאריך שנקבע בחוזה
בעבור כל חודשי האיחור, כולל החודשיים המותרים לאיחור על פי החוק. על המוכר לפצות את הרוכש 
חודש בחודשו לפי שווי של 150% בו נרכשה שמשכר דירה המ ...

#2 | score: 0.6749 | page: 15 | id: chunk-19
סביר מראש. רצוי לקבל 
זהותאת הַחֲזָקָה בדירה לאחר שטיפת הרצפה וניקוי הכלים הסניטריים על ידי המוכר, כדי שתוכלו ל אם 
קיימים פגמים. הקפידו לערוך בדיקה יסודית של הדירה לפני קבלת החזקה על ידיכם או על ידי בא כוח 
מקצועי מטעמכם – רצוי מהנדס . 
דאגו לערוך פרוטוקול מסירה ברור ומפורט אשר יכלול את פירוט כל הל ...

#3 | score: 0.6502 | page: 14 | id: chunk-18
כי "דרך המלך" 
לביצוע תשלום היא בהגעה פיזית לפקיד הבנק. למרות זאת, ניתן לבצע תשלום זה ללא הגעה בדרך 
ההקבוע להלן קישור לנייר העמדה בנייר עמדה אשר פרסם משרד הבינוי והשיכון. – "תשלום מר

### Conclusions (to be filled after running)

**What to expect:**
- In setting B (smaller chunks) – **more chunks** are created, and the top result is often **more focused** around the compensation/delay topic.
- In setting A (larger chunks) – fewer chunks, and sometimes the result includes **more context** (additional information not directly relevant to the question).
- Higher overlap (150 vs 80) – reduces the chance that a sentence is cut in the middle, at the cost of redundant chunks.

**Short notes (values after you run it):**

| | Setting A (1000/150) | Setting B (500/80) |
|---|---|---|
| Number of chunks | 24 | 48 |
| Score of result #1 | 0.6932  | 0.7266 |
| Is the result relevant? | YES | YES |

> **Recommendation for the project:** `chunk_size=1000, overlap=150` works well for a medium-length Hebrew PDF document.
> If retrieval is too broad, try reducing it to 600–800.